# Post-Pandemic Gender Labor Force Trends
Principle Investigators: Abigail Garvey, Anna Wang, Rachelle Lang - Group Q

In [1]:
# uncomment if not installed - needed to save the table as an svg
!pip install dataframe-image matplotlib

In [2]:
import pandas as pd
import statsmodels.formula.api as smf
import datetime as dt
import dataframe_image as dfi

In [3]:
#read in industries data and change the date column to type datetime
industries = pd.read_csv("../Data/industries.csv")
industries["date"] =  pd.to_datetime(industries["date"])

In [4]:
#find all the unique industry codes
industry_codes = industries["indy_code"].unique()
# names of the industries in the dataset
industry_names = ["All Industries", "Agriculture and related industries", "Agriculture, forestry, fishing, and hunting", "Nonagriculture industries",
              "Mining, quarrying, and oil and gas extraction", "Utilities", "Construction", "Nondurable goods manufacturing",
              "Manufacturing", "Durable goods manufacturing", "Wholesale and retail trade", "Wholesale trade", "Retail trade",
              "Transportation and utilities", "Transportation and warehousing", "Information", "Financial activities",
             "Finance and insurance", "Real estate and rental and leasing", "Professional and business services", "Professional and technical services",
             "Management, administrative, and waste services", "Education and health services", "Educational services",
             "Health care and social assistance","Health service, except hospitals","Hospitals","Social assistance",
             "Leisure and hospitality","Arts, entertainment, and recreation", "Accommodation and food services", "Other services",
             "Other services, except private households", "Other services, private households", "Public administration"]

# map the codes to the names with a dictionary of the names
indy_code_to_name = dict(zip(industry_codes, industry_names))
industries['industry_name'] = industries['indy_code'].map(indy_code_to_name)

In [5]:
'''
used https://medium.com/@JuanPabloHerrera/why-interrupted-time-series-might-be-your-new-favorite-data-tool-399d260a9cd9
as an example of how to do interrupted time series analysis with regression
'''
# creating a mask for if the date is before or after covid
covid_start = pd.to_datetime("2020-03-01")
mask = industries['date'] > covid_start

# time column to show the number of months since the first month in the data, 1983-01-01
industries["time"] = (
    industries
    .groupby('indy_code')
    .cumcount()
)

# time_since_covid column to show the number of months since covid
industries["time_since_covid"] =  (
    industries[mask]
    .groupby('indy_code')
    .cumcount()
)

industries["post_covid"] = (industries["date"] >= covid_start).astype(int)
industries['time_since_covid'] = industries['time_since_covid'].fillna(0)

In [6]:
industries.head()

,indy_code,occupation_code,date,employed_men,employed_women,employed_total,pct_men,pct_women,industry_name,time,time_since_covid,post_covid
0,0,7,1983-01-01,16410.0,12123.0,28533.0,0.575124,0.424876,All Industries,0,0.0,0
1,0,7,1983-02-01,16355.0,12225.0,28580.0,0.572253,0.427747,All Industries,1,0.0,0
2,0,7,1983-03-01,16491.0,12201.0,28692.0,0.574760,0.425240,All Industries,2,0.0,0
3,0,7,1983-04-01,16759.0,12366.0,29125.0,0.575416,0.424584,All Industries,3,0.0,0
4,0,7,1983-05-01,16825.0,12367.0,29192.0,0.576357,0.423643,All Industries,4,0.0,0


In [7]:
# filtering the industries to just include 5 years pre and post covid
filteredIndustries = industries[(industries["date"] >= dt.datetime(2015, 3, 1)) & (industries["date"] <= dt.datetime(2025, 3, 1))]

In [8]:
results = {}

# run an ols regression on each of the industries in the filteredIndustries dataset and uses robust standard errors
for industry, group in filteredIndustries.groupby('industry_name'):
    model = smf.ols("employed_total ~ time + post_covid + C(occupation_code) + time_since_covid", data=group).fit(cov_type="HC3")
    df_result = pd.DataFrame({
        "coef": model.params,
        "pvalue": model.pvalues
    })
    results[industry] = df_result


In [9]:
# turn all the regression results into a dataframe
results = pd.concat(results, names=["industry_name", "variable"])
results = results.reset_index()
results

,industry_name,variable,coef,pvalue
0,Accommodation and food services,Intercept,8146.596374,6.202449e-14
1,Accommodation and food services,time,6.724999,1.044773e-02
2,Accommodation and food services,post_covid,-2219.594202,7.618754e-14
3,Accommodation and food services,time_since_covid,31.625314,6.559766e-05
4,Agriculture and related industries,Intercept,2597.695455,0.000000e+00
...,...,...,...,...
368,Wholesale trade,C(occupation_code)[T.7699],-2114.520391,2.290422e-17
369,Wholesale trade,C(occupation_code)[T.8999],-1439.937719,1.438873e-07
370,Wholesale trade,time,-0.623928,4.504722e-07
371,Wholesale trade,post_covid,-78.686599,2.474464e-12


In [10]:
# These industries had a significant negative time_since_covid pvalue
filteredNegInd = results[(results["pvalue"] < 0.05) & (results["coef"] < 0) & (results["variable"] == "time_since_covid")]
filteredNegInd = filteredNegInd.drop(["variable"], axis=1)
filteredNegInd = filteredNegInd.rename(columns={'industry_name': 'Industry', 'pvalue': 'p-value'})
filteredNegInd.reset_index(drop=True, inplace = True)

In [11]:
filteredNegInd

,Industry,coef,p-value
0,All Industries,-1.699006,4.571251e-94
1,Construction,-0.570648,8.109740e-21
2,Durable goods manufacturing,-0.180732,1.297836e-05
3,Education and health services,-1.032716,1.083943e-14
4,Financial activities,-0.498793,1.164736e-44
5,Hospitals,-7.183301,1.085244e-04
6,Information,-0.059068,1.639361e-03
7,Manufacturing,-0.233386,5.294496e-05
8,Nondurable goods manufacturing,-0.052836,3.495780e-02
9,Professional and business services,-1.523461,2.891103e-45


In [12]:
# Saving the dataframe as an svg
filename = 'negative_trends_industries.svg'
dfi.export(filteredNegInd, filename, table_conversion='matplotlib')

print(f"Successfully saved DataFrame to {filename}")

Successfully saved DataFrame to negative_trends_industries.svg


In [13]:
# These industries had a significant positive time_since_covid pvalue
filteredPosInd = results[(results["pvalue"] < 0.05) & (results["coef"] > 0) & (results["variable"] == "time_since_covid")]
filteredPosInd = filteredPosInd.drop(["variable"], axis=1)
filteredPosInd = filteredPosInd.rename(columns={'industry_name': 'Industry', 'pvalue': 'p-value'})
filteredPosInd.reset_index(drop=True, inplace = True)

In [14]:
filteredPosInd

,Industry,coef,p-value
0,Accommodation and food services,31.625314,6.559766e-05
1,Agriculture and related industries,0.092501,2.459063e-15
2,"Arts, entertainment, and recreation",14.570427,3.315552e-07
3,Educational services,10.920603,2.850981e-04
4,Health care and social assistance,19.880557,5.388884e-06
5,"Health service, except hospitals",15.254989,1.345643e-07
6,Leisure and hospitality,0.723205,3.581095e-09
7,"Management, administrative, and waste services",28.462852,3.923042e-17
8,"Mining, quarrying, and oil and gas extraction",0.081893,4.699305e-14
9,Nonagriculture industries,138.567645,9.781431e-05


In [15]:
# saving the dataframe as an SVG
filename = 'positive_trends_industries.svg'
dfi.export(filteredPosInd, filename, table_conversion='matplotlib')

print(f"Successfully saved DataFrame to {filename}")

Successfully saved DataFrame to positive_trends_industries.svg
